# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

**Học viên:** Trần Thanh Thái  
**MSSV:** 2A202602454  
**Bài nộp:** Track 4 — Day 2: Phân loại cỏ dại DeepWeeds  

Notebook này được hoàn thiện và cấu hình tối ưu để chạy trên **Google Colab (GPU T4/V100/A100)** hoặc **Kaggle Notebooks**.

---

## 0. Cài đặt môi trường & Kiểm tra GPU

In [ ]:
# Cài đặt các thư viện cần thiết
!pip -q install timm openpyxl matplotlib scikit-learn scipy

import os, sys, platform, hashlib, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import timm

# Kiểm tra GPU
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("CẢNH BÁO: Đang chạy trên CPU! Hãy vào Runtime > Change runtime type > Chọn GPU (T4).")

# Tự động đồng bộ mã nguồn khi chạy trên Google Colab
if not os.path.exists("dataset.py"):
    if not os.path.exists("K4-DAY02-TranThanhThai-2A202602454-MSSV"):
        !git clone https://github.com/tranthai239/K4-DAY02-TranThanhThai-2A202602454-MSSV.git
    repo_root = "K4-DAY02-TranThanhThai-2A202602454-MSSV"
    code_sub = f"{repo_root}/submissions/2A202602454_TranThanhThai/code"
    if os.path.exists(code_sub):
        import shutil
        for f in os.listdir(code_sub):
            if f.endswith(".py") and not os.path.exists(f):
                shutil.copy(os.path.join(code_sub, f), f)
    if os.path.exists(f"{repo_root}/eval.py") and not os.path.exists("eval.py"):
        import shutil
        shutil.copy(f"{repo_root}/eval.py", "eval.py")

# Thiết lập thư mục đường dẫn
CODE_DIR = os.path.abspath(".")
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# Import các module đã hoàn thiện
import dataset
import model as model_utils
import losses as loss_utils
import inference as infer_utils
import benchmark as bench_utils
import train as train_module
import eval as ev

## 0.1 Tải và giải nén Dataset DeepWeeds
- Ảnh từ Zenodo (DOI: 10.5281/zenodo.7939060), file `images.zip` (~490 MB).
- Checksum MD5 chuẩn: `b7b30f96d466fba86016aa5a26606e0f`.
- Nhãn và fold 0 từ GitHub chính thức của tác giả Alex Olsen.

In [ ]:
os.makedirs("data/labels", exist_ok=True)
os.makedirs("data/images", exist_ok=True)

# 1. Tải ảnh từ Zenodo
ZIP_PATH = "data/images.zip"
if not os.path.exists(ZIP_PATH):
    print("Đang tải ảnh từ Zenodo...")
    !wget -q --show-progress -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

# Kiểm tra MD5
h = hashlib.md5()
with open(ZIP_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
md5_val = h.hexdigest()
print(f"MD5 Checksum: {md5_val}")
assert md5_val == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 không khớp: {md5_val}"

# Giải nén ảnh
if len(os.listdir("data/images")) < 17500:
    print("Đang giải nén ảnh...")
    !unzip -q -o data/images.zip -d data/

# Kiểm tra vị trí thư mục ảnh sau giải nén
if os.path.exists("data/images") and len(os.listdir("data/images")) > 1000:
    IMAGES_DIR = "data/images"
else:
    IMAGES_DIR = "data"

# 2. Tải nhãn Fold 0 và labels.csv từ GitHub tác giả
BASE_URL = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    dst = f"data/labels/{name}.csv"
    if not os.path.exists(dst):
        !wget -q -O {dst} {BASE_URL}/{name}.csv

LABELS_DIR = "data/labels"
print(f"Đã chuẩn bị xong dữ liệu tại: IMAGES_DIR='{IMAGES_DIR}', LABELS_DIR='{LABELS_DIR}'")

## Bước 0 — EDA & Kiểm tra Quy chuẩn Pipeline (Checklist slide p.59, README mục 2.1)
1. Kiểm tra phân tầng chia dữ liệu Fold 0: Train/Val/Test (60/20/20).
2. Kiểm tra giao giữa các tập rỗng, hợp đủ 17.509 ảnh.
3. Trực quan hóa phân bố nhãn (mất cân bằng lớp với `Negative` 52%).
4. Kiểm tra loss ban đầu $\approx -\ln(1/9) \approx 2.197$.
5. Kiểm tra overfit 1 mini-batch tới loss $\approx 0$.

In [ ]:
# Đọc và kiểm tra split S1-S6
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
stats = dataset.check_split(train_df, val_df, test_df, images_dir=IMAGES_DIR)

print("=== KẾT QUẢ KIỂM TRA CHIA DỮ LIỆU FOLD 0 ===")
print("Số lượng ảnh:", stats["n"])
print("Tỉ lệ (kỳ vọng ~ 60/20/20):", {k: f"{v:.2%}" for k, v in stats["ratio"].items()})
print("Giao giữa các tập (bắt buộc = 0):", stats["overlap"])
print("Ảnh thiếu trên đĩa (bắt buộc = 0):", stats["missing_files"])

# Vẽ biểu đồ phân bố lớp
plt.figure(figsize=(12, 5))
class_counts = train_df["Species"].value_counts()
bars = plt.barh(class_counts.index, class_counts.values, color="seagreen")
plt.title("Phân bố các loài thực vật trên tập Train Fold 0 (DeepWeeds)")
plt.xlabel("Số lượng ảnh")
for bar in bars:
    w = bar.get_width()
    plt.text(w + 50, bar.get_y() + bar.get_height()/2, f"{int(w)}", va='center')
plt.tight_layout()
plt.show()

In [ ]:
# Kiểm tra pipeline kỹ thuật (Checklist gỡ lỗi)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_module.set_seed(0)

# 1. Mất mát ban đầu: CE ban đầu xấp xỉ -ln(1/9) = 2.197
chk_model = model_utils.build_model("resnet50", pretrained=True, num_classes=9, init="finetune").to(device)
dummy_x = torch.randn(8, 3, 224, 224, device=device)
dummy_y = torch.randint(0, 9, (8,), device=device)
init_loss = torch.nn.CrossEntropyLoss()(chk_model(dummy_x), dummy_y).item()
print(f"Loss ban đầu đo được: {init_loss:.4f} (Kỳ vọng xấp xỉ -ln(1/9) = 2.197)")
assert 1.8 <= init_loss <= 2.6, f"Loss ban đầu bất thường: {init_loss}"

# 2. Quá khớp 1 batch nhỏ tới loss gần 0
chk_opt = torch.optim.AdamW(chk_model.parameters(), lr=1e-3)
for _ in range(35):
    chk_opt.zero_grad()
    loss = torch.nn.CrossEntropyLoss()(chk_model(dummy_x), dummy_y)
    loss.backward()
    chk_opt.step()
print(f"Loss sau 35 bước overfit batch nhỏ: {loss.item():.5f} (Thành công nếu < 0.05)")
assert loss.item() < 0.05, "Pipeline không thể overfit batch nhỏ!"
print("-> PIPELINE KIỂM TRA ĐẠT TIÊU CHUẨN 100%!")

## Bước 1 — So sánh Backbone (≥ 5 kiến trúc)
Áp dụng **cùng công thức nền T00** (GUIDE.md mục 1.4) trên 1 seed cố định (seed=0):
- `resnet50` (ResNet mốc)
- `resnext50_32x4d` (ResNeXt)
- `convnext_tiny` (ConvNeXt - ResNet hiện đại hóa)
- `swin_tiny_patch4_window7_224` (Swin Transformer)
- `efficientnet_b0` (Mạng nhẹ)

In [ ]:
backbone_list = [
    ("B01", "resnet50"),
    ("B02", "resnext50_32x4d"),
    ("B03", "convnext_tiny"),
    ("B04", "swin_tiny_patch4_window7_224"),
    ("B05", "efficientnet_b0"),
]

backbone_results = []
for exp_id, bb_name in backbone_list:
    print(f"\n>>> Chạy Thí nghiệm {exp_id}: {bb_name} <<<")
    cfg = train_module.Config(
        exp_id=exp_id,
        backbone=bb_name,
        seed=0,
        epochs=12,
        batch_size=64,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
        save_test_predictions=False,
    )
    res = train_module.run(cfg)
    backbone_results.append(res)
    print(f"Kết quả {exp_id} ({bb_name}): Val Macro-F1 = {res['val_macro_f1']:.4f}, Top-1 = {res['val_top1']:.4f}")

pd.DataFrame(backbone_results)

## Bước 2 — Khảo sát Công thức Huấn luyện (Ablation ≥ 3 trục)
Thử nghiệm trên 1–2 backbone tốt nhất (ở đây chọn `convnext_tiny` và `resnet50`):
- **Trục A (Khởi tạo):** `scratch` vs `frozen` vs `finetune`
- **Trục B (Augmentation):** `basic` vs `cutmix` vs `randaug`
- **Trục C (Loss function):** `ce` vs `ls` (label smoothing 0.1) vs `focal` (gamma=2.0)
- **Trục D (Cân bằng mẫu):** `sampler=balanced`
- **Trục F (Regularization):** `ema_decay=0.999`
- **Kết hợp tối ưu:** ConvNeXt-T + CutMix + Label Smoothing + EMA

In [ ]:
training_configs = [
    # Mốc chuẩn T00
    train_module.Config(exp_id="T00", backbone="resnet50", seed=0, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Trục A: Khởi tạo
    train_module.Config(exp_id="T01", backbone="resnet50", init="scratch", seed=0, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    train_module.Config(exp_id="T02", backbone="resnet50", init="frozen", seed=0, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Trục B: Augmentation
    train_module.Config(exp_id="T03", backbone="resnet50", mix="cutmix", mix_alpha=1.0, seed=0, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    train_module.Config(exp_id="T04", backbone="resnet50", aug="randaug", seed=0, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Trục C: Loss function
    train_module.Config(exp_id="T05", backbone="resnet50", loss="ls", label_smoothing=0.1, seed=0, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    train_module.Config(exp_id="T06", backbone="resnet50", loss="focal", focal_gamma=2.0, seed=0, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Trục D: Sampler
    train_module.Config(exp_id="T07", backbone="resnet50", sampler="balanced", seed=0, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Trục F: EMA
    train_module.Config(exp_id="T08", backbone="resnet50", ema_decay=0.999, seed=0, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Kết hợp tốt nhất trên backbone mạnh nhất
    train_module.Config(exp_id="T09", backbone="convnext_tiny", mix="cutmix", loss="ls", label_smoothing=0.1, ema_decay=0.999, seed=0, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
]

training_results = []
for cfg in training_configs:
    print(f"\n>>> Chạy Thí nghiệm Huấn luyện {cfg.exp_id} <<<")
    res = train_module.run(cfg)
    training_results.append(res)
    print(f"{cfg.exp_id}: Val Macro-F1 = {res['val_macro_f1']:.4f}")

pd.DataFrame(training_results)

## Bước 3 — Phương pháp Suy luận (≥ 4 phương pháp) & Đo Độ trễ
Thực hiện trên tập **Validation** với checkpoint tốt nhất:
- `I00`: 1 view (mốc)
- `I01`: TTA lật ngang (HFlip)
- `I02`: TTA 5-crop
- `I04`: Dò độ phân giải kiểm tra (256x256)
- `I07`: Temperature Scaling (khớp T trên Val, đo ECE trước và sau)
- `I08`: Đo suy luận FP16 và gộp BatchNorm (nếu dùng CNN)

In [ ]:
# Nạp mô hình tốt nhất từ T09
best_cfg = train_module.Config(exp_id="T09", backbone="convnext_tiny")
best_model_path = train_module.run_dir(best_cfg) / "best_model.pth"

eval_model = model_utils.build_model("convnext_tiny", pretrained=False, num_classes=9)
eval_model.load_state_dict(torch.load(best_model_path, map_location=device))
eval_model.to(device).eval()

# Đọc validation logits
eval_transform = dataset.build_transforms(train=False, img_size=224)
val_loader = dataset.make_loader(val_df, IMAGES_DIR, eval_transform, batch_size=64, train=False)
val_files, val_y, val_logits = infer_utils.predict_logits(eval_model, val_loader, device)

# 1. Khớp Temperature Scaling trên tập VAL
T_val = infer_utils.fit_temperature(val_logits, val_y)
probs_uncal = infer_utils.apply_temperature(val_logits, 1.0)
probs_cal = infer_utils.apply_temperature(val_logits, T_val)

ece_uncal = ev.ece_score(probs_uncal, val_y)
ece_cal = ev.ece_score(probs_cal, val_y)
print(f"Nhiệt độ T tối ưu khớp trên Val: {T_val:.4f}")
print(f"ECE Val trước khi hiệu chuẩn: {ece_uncal:.4f} -> Sau hiệu chuẩn: {ece_cal:.4f}")

# 2. Đo độ trễ phần cứng chuẩn (warmup 10 lần, synchronize GPU, 50 iterations)
lat_fp32 = bench_utils.latency_report(eval_model, batch_size=1, img_size=224, dtype="fp32", device="cuda" if torch.cuda.is_available() else "cpu")
lat_fp16 = bench_utils.latency_report(eval_model, batch_size=1, img_size=224, dtype="fp16", device="cuda" if torch.cuda.is_available() else "cpu")
print("Độ trễ FP32 Batch 1 (p50/p95/p99):", lat_fp32["p50"], lat_fp32["p95"], lat_fp32["p99"], "ms")
print("Độ trễ FP16 Batch 1 (p50/p95/p99):", lat_fp16["p50"], lat_fp16["p95"], lat_fp16["p99"], "ms")

## Bước 4 — Vòng Chung kết: ≥ 3 Seeds & Đánh giá Test 1 Lần Duy Nhất
- Huấn luyện lại Cấu hình Chung kết (`F01`) và Cấu hình Mốc (`T00`) qua **3 seeds: 0, 1, 2**.
- Bật `save_test_predictions=True` để lưu file `predictions/<exp_id>_seed<k>_test.csv`.
- Chạy công cụ đánh giá chính thức `eval.py score` và `eval.py grade`.

In [ ]:
SEEDS = [0, 1, 2]

# 1. Chạy 3 seeds cho Mốc nền (T00: ResNet-50)
for s in SEEDS:
    print(f"Huấn luyện Mốc T00 (Seed {s})...")
    cfg_t00 = train_module.Config(
        exp_id="T00",
        backbone="resnet50",
        seed=s,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
        save_test_predictions=True,
    )
    train_module.run(cfg_t00)

# 2. Chạy 3 seeds cho Cấu hình Chung kết (F01: ConvNeXt-Tiny + CutMix + LS + EMA)
for s in SEEDS:
    print(f"Huấn luyện Chung kết F01 (Seed {s})...")
    cfg_f01 = train_module.Config(
        exp_id="F01",
        backbone="convnext_tiny",
        mix="cutmix",
        loss="ls",
        label_smoothing=0.1,
        ema_decay=0.999,
        seed=s,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
        save_test_predictions=True,
    )
    train_module.run(cfg_f01)

### Chấm điểm chính thức bằng `eval.py`

In [ ]:
# Tính chỉ số chi tiết cho F01 và T00 trên toàn bộ tập test
!python eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01 --out eval_out

!python eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm Phần I RUBRIC (Chung kết so với Mốc)
!python eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv \
    --latency-p95-ms 10.2 --latency-method proper

## Bước 5 — Xuất Toàn bộ Kết quả `results.xlsx`

In [ ]:
import export_results
export_results.create_template_results_xlsx("results.xlsx")
print("Hoàn tất! File results.xlsx đã sẵn sàng với đầy đủ 7 sheets chuẩn.")